# Lesson 01 - บทนำเกี่ยวกับ AI Agents

ยินดีต้อนรับสู่บทเรียนแรกในหลักสูตร **AI Agents สำหรับผู้เริ่มต้น**!

**AI agent** คือโปรแกรมที่ใช้โมเดลภาษาขนาดใหญ่ (LLM) เป็นเครื่องมือในการคิดวิเคราะห์ และสามารถดำเนิน *การกระทำ* ในโลกจริงได้ — เช่น การเรียกใช้ API, การสอบถามฐานข้อมูล หรือการรันโค้ด — เพื่อบรรลุเป้าหมายแทนผู้ใช้

ในไฟล์โน้ตบุ๊กนี้ คุณจะสร้างเอเย่นต์ตัวแรกของคุณ: **Travel Agent** ที่แนะนำสถานที่ท่องเที่ยว ในระหว่างทางคุณจะได้เรียนรู้วิธี:

1. เชื่อมต่อกับ Azure AI Foundry Agent Service โดยใช้ **Microsoft Agent Framework**
2. ให้เอเย่นต์มี **เครื่องมือ** — ฟังก์ชัน Python ธรรมดาที่เอเย่นต์สามารถเรียกใช้ได้
3. รันเอเย่นต์และตรวจสอบการตอบสนองของมัน
4. สตรีมการตอบสนองของเอเย่นต์ทีละโทเค็น


## Setup

ก่อนที่จะรันโน้ตบุ๊กนี้ ให้แน่ใจว่าคุณมี:

1. **โปรเจกต์ Azure AI Foundry** ที่มีโมเดลแชทที่ถูกติดตั้ง (เช่น `gpt-4o-mini`)
2. **เข้าสู่ระบบด้วย Azure CLI** — รันคำสั่ง `az login` ในเทอร์มินัลของคุณ
3. **ตั้งค่าตัวแปรสภาพแวดล้อมที่จำเป็น:**
   - `AZURE_AI_PROJECT_ENDPOINT` — จุดสิ้นสุดโปรเจกต์ Azure AI Foundry ของคุณ
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อของโมเดลที่คุณติดตั้ง

เซลล์ด้านล่างจะติดตั้งแพ็กเกจ Python ที่คุณต้องการใช้ฟรี


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## การสร้างเอเจนต์ตัวแรกของคุณ

เอเจนต์ต้องการสิ่งสองอย่าง:

- **คำสั่ง** ที่บอกมันว่า *มันคือใคร* และ *จะประพฤติตัวอย่างไร* (พรอมต์ระบบ)
- **เครื่องมือ** — ฟังก์ชัน Python ที่ถูกตกแต่งด้วย `@tool` ซึ่งเอเจนต์สามารถเรียกใช้เพื่อดึงข้อมูลหรือทำงานต่างๆ

ด้านล่างเราจะกำหนดเครื่องมืออย่างง่ายที่คืนรายชื่อจุดหมายปลายทางยอดนิยมสำหรับการพักผ่อน เอเจนต์จะใช้เครื่องมือนี้เมื่อผู้ใช้ขอคำแนะนำการเดินทาง


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

สำหรับประสบการณ์ที่โต้ตอบได้มากขึ้น คุณสามารถ **สตรีม** การตอบกลับของเอเจนต์ได้ แทนที่จะรอคำตอบทั้งหมด เอเจนต์จะส่งข้อความเป็นชิ้น ๆ ขณะที่สร้างขึ้น สิ่งนี้มีประโยชน์อย่างยิ่งในอินเทอร์เฟซแชทที่คุณต้องการแสดงผลลัพธ์แบบเรียลไทม์


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธี:

- **สร้างผู้ให้บริการ** ที่เชื่อมต่อกับ Azure AI Foundry Agent Service ผ่าน `AzureAIProjectAgentProvider`
- **กำหนดเครื่องมือ** โดยใช้ตัวตกแต่ง `@tool` เพื่อให้เอเย่นต์สามารถเรียกใช้ฟังก์ชัน Python ของคุณได้
- **เรียกใช้งานเอเย่นต์** พร้อมข้อความจากผู้ใช้และพิมพ์การตอบกลับของมัน
- **สตรีมการตอบกลับ** เพื่อแสดงผลแบบเรียลไทม์

ในบทเรียนถัดไปเราจะสำรวจกรอบงานเอเย่นต์อย่างลึกซึ้งมากขึ้นและเรียนรู้วิธีให้เอเย่นต์มีเครื่องมือที่ทรงพลังขึ้นและความสามารถในการให้เหตุผลหลายขั้นตอน


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
